In [128]:
!pip install PyMuPDF


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [129]:
import fitz  # PyMuPDF
import re
import json
from pathlib import Path
from pprint import pprint
from typing import List, Dict

In [130]:
bns_path = Path.cwd() / "documents" / "BNS.pdf"
bns_doc = fitz.open(bns_path.as_posix())
output_dir = Path.cwd() / "contents"

In [131]:
def extract_bns_sections(pdf_path: str) -> List[Dict]:
    """
    Extract numbered sections (1–358) from the given PDF.
    This version reads text line by line and captures section content robustly.
    """

    # ---- 1) Read PDF lines ----

    doc = fitz.open(pdf_path)
    lines = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text")
        # Normalize CR/LF
        text = text.replace("\r\n", "\n").replace("\r", "\n").replace("_", "")
        page_lines = text.split("\n")
        lines.extend(page_lines)

    doc.close()

    # ---- 2) Regex for section header lines ----

    # Matches at start of line, e.g.:
    #   "1."
    #   "104."
    #   "25 ."
    header_re = re.compile(r"^\s*(\d{1,3})\s*\.\s*$")

    # Sometimes the number and text appear on the same line:
    header_with_text_re = re.compile(r"^\s*(\d{1,3})\s*\.\s*(.+)$")

    sections = {}
    current_section = None

    # ---- 3) Iterate lines and group content ----

    for line in lines:

        # Try match "number + text on same line"
        m_full = header_with_text_re.match(line)
        if m_full:
            num = int(m_full.group(1))
            text = m_full.group(2).strip()

            # Start a new section
            current_section = num
            sections[current_section] = text
            continue

        # Try match "only number on line"
        m_num = header_re.match(line)
        if m_num:
            num = int(m_num.group(1))
            current_section = num
            sections[current_section] = ""
            continue

        # If we are inside a section, append to its content
        if current_section is not None:
            # Append text with a space
            existing = sections[current_section]
            if existing:
                sections[current_section] = existing + " " + line.strip()
            else:
                sections[current_section] = line.strip()

    # ---- 4) Build ordered list 1–358 ----

    output = []
    for num in range(1, 359):
        content = sections.get(num, "")
        # Normalize whitespace
        content = re.sub(r"\s+", " ", content).strip()
        output.append({"section_number": num, "content": content})

    return output


def save_as_json(data, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

In [132]:
s = extract_bns_sections(bns_path.as_posix())
save_as_json(s, output_dir / "bns_sections.json")